In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import xml.etree.ElementTree as ET
import numpy as np


# Scraping eu-site
**What should be included:**
- Names (rappertour and shadow)
- Field
- Political party
- Document text

Names, field, document code and political party can be scraped from: https://oeil.secure.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024

Shadow rapporteur from: https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?
- The document code should be inputtet, and can be fetched from the previous

Context of the document comes from: https://oeil.secure.europarl.europa.eu/oeil/en/document-summary?id=1785947
- Id can be found by scraping the above web page, stored under Legislative proposal


In [3]:
#Finding out how it can be accesed
link = "https://oeil.secure.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&reference.type=legAct&reference.initialType=legAct&term=9th+term+2019+-+2024&resultsOnly=true"
r = requests.get(link)
soup = BeautifulSoup(r.content)
names = soup.find_all("span", class_="erpl_document-subtitle-author")
rapporteurs = [name.text.strip() for name in names]
rapporteurs_out = [", ".join(rapporteurs)]
rapporteurs_out

['CUNHA Paulo (EPP), GONZÁLEZ CASARES Nicolás (S&D), KELLER Fabienne (Renew), GREGOROVÁ Markéta (Greens/EFA), KALNIETE Sandra (EPP), SIPPEL Birgit (S&D), NEMEC Matjaž (S&D), RESSLER Karlo (EPP), PICULA Tonino (S&D), MANDERS Antonius (EPP), KUHNKE Alice (Greens/EFA), AGUILERA Clara (S&D), BALLARÍN CEREZA Laura (S&D), CAVAZZINI Anna (Greens/EFA), SINČIĆ Ivan Vilibor (NI), VOSS Axel (EPP), OETJEN Jan-Christoph (Renew), OETJEN Jan-Christoph (Renew), LÓPEZ AGUILAR Juan Fernando (S&D), SINČIĆ Ivan Vilibor (NI), ĎURIŠ NICHOLSONOVÁ Lucia (Renew), BIELAN Adam (ECR), MORTLER Marlene (EPP), GUERREIRO Francisco (Greens/EFA), VAN OVERTVELDT Johan (ECR), NEMEC Matjaž (S&D), HAUTALA Heidi (Greens/EFA), BORCHIA Paolo (ID), GAHLER Michael (EPP), GARDIAZABAL RUBIAL Eider (S&D)']

In [4]:
commiteer = soup.find_all("span", class_="erpl_badge-committee")
committee_title = [c.get("title") for c in commiteer if c.has_attr("title")]
committee_title

['Civil Liberties, Justice and Home Affairs',
 'Agriculture and Rural Development',
 'Industry, Research and Energy',
 'Civil Liberties, Justice and Home Affairs',
 'International Trade',
 'International Trade',
 'Environment, Climate and Food Safety',
 'Economic and Monetary Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Foreign Affairs',
 'Budgets',
 'Employment and Social Affairs',
 'Civil Liberties, Justice and Home Affairs',
 'Agriculture and Rural Development',
 'Internal Market and Consumer Protection',
 'Internal Market and Consumer Protection',
 'Environment, Public Health and Food Safety',
 'Legal Affairs',
 'Transport and Tourism',
 'Transport and Tourism',
 'Civil Liberties, Justice and Home Affairs',
 'Environment, Public Health and Food Safety',
 'Employment and Social Affairs',
 'Internal Market and Consumer Protection',
 'Environment, Public Health and Food Safety',
 'Fisheries',
 'Economic and Monetary Affairs',


In [8]:
# URL  - her kun sorteret for emne Agriculture, hvis datasættet skal udvides kan man bare søge efter andre emner og så gemme i seperate datasæt
# som til sidst kan sættes sammen (det tror jeg er nemmest)
url = "https://oeil.secure.europarl.europa.eu/oeil/en/search/export/XML?fullText.mode=EXACT_WORD&term=9th+term+2019+-+2024&subject=3.10+Agricultural+policy+and+economies+&resultsOnly=true"


response = requests.get(url)
response.raise_for_status()

root = ET.fromstring(response.content)


data = []

for item in root.find("items").findall("item"):
    reference = item.findtext("reference", default="")
    title = item.findtext("title", default="")
    
    # Extract all rapporteurs 
    rapporteurs_elem = item.find("rapporteur")
    if rapporteurs_elem is not None:
        rapporteurs = [r.text for r in rapporteurs_elem.findall("rapporteur") if r.text]
    else:
        rapporteurs = ['NaN']

    rapporteurs_str = ", ".join(rapporteurs) if rapporteurs else ""

    data.append({
        "document_id": reference,
        "title": title,
        "rapporteurs": rapporteurs_str,
    })

df = pd.DataFrame(data)
print(df.head())


      document_id                                              title  \
0  2024/0144(COD)  Economic accounts for agriculture in the Union...   
1  2024/0073(COD)  Common Agricultural Policy (CAP): good agricul...   
2  2024/0030(COD)  Equivalence of field inspections carried out i...   
3  2024/0027(COD)  Granting equivalence with EU requirements to M...   
4  2023/0448(COD)  Protection of animals during transport and rel...   

                                  rapporteurs  
0                                              
1                                              
2                                              
3                   VRECIONOVÁ Veronika (ECR)  
4  BUDA Daniel (EPP), METZ Tilly (Greens/EFA)  


In [9]:
df['rapporteurs'] = df['rapporteurs'].replace('', np.nan) 
df

,document_id,title,rapporteurs
0,2024/0144(COD),Economic accounts for agriculture in the Union...,NaN
1,2024/0073(COD),Common Agricultural Policy (CAP): good agricul...,NaN
2,2024/0030(COD),Equivalence of field inspections carried out i...,NaN
3,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)
4,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)"
...,...,...,...
381,2019/2776(RPS),Resolution on the draft Commission regulation ...,"ANDRIEU Eric (S&D), HOJSÍK Martin (Renew), EIC..."
382,COM(2024)0225,Force majeure and exceptional circumstances in...,NaN
383,COM(2024)0194,International Olive Council (IOC): two methods...,NaN
384,COM(2024)0168,"International Sugar Agreement, 1992: condition...",NaN


In [11]:
#removing nan
df_agri = df.dropna()
df_agri.reset_index(drop=True, inplace= True)
df_agri

,document_id,title,rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)
1,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)"
2,2023/0447(COD),Welfare of dogs and cats and their traceability,VRECIONOVÁ Veronika (ECR)
3,2023/0413(COD),Monitoring framework for resilient European fo...,"SARGIACOMO Eric (S&D), WIESNER Emma (Renew)"
4,2023/0410(COD),Standing Forest and Forestry Expert Group,"TOVERI Pekka (EPP), WIESNER Emma (Renew)"
...,...,...,...
83,2023/2726(RPS),Commission Regulation amending Annex II to Reg...,RIVASI Michèle (Greens/EFA)
84,2021/2608(RPS),Resolution on the draft Commission regulation ...,RIVASI Michèle (Greens/EFA)
85,2020/2795(RPS),Resolution on the draft Commission regulation ...,"NOVAK Ljudmila (EPP), ANDRIEU Eric (S&D), RIVA..."
86,2020/2735(RPS),Resolution on the draft Commission regulation ...,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ..."


# Find shadow rapporteurs and document url

In [26]:
df_test = df_agri.copy()
df_test = df_test[:1]
df_test

,document_id,title,rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR)


In [33]:
def get_shadow_rapporteurs(reference):
    base_url = "https://oeil.secure.europarl.europa.eu/oeil/en/procedure-file?reference="
    url = base_url + reference

    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch: {url}")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    try:
        shadow_section = soup.find("div", id="collapseShadowRapporteur")
        if not shadow_section:
            return None
        
        shadow_names = []
        for a in shadow_section.find_all("a", class_= "rapporteur"):
            span = a.find("span")
            #print("span", span)
            if span:
                shadow_names.append(span.get_text(strip=True))
                #print("shadow names:", shadow_names)

        return ", ".join(shadow_names) if shadow_names else None
    
    except Exception as e:
        print(f"Error parsing {reference}: {e}")
        return None

df_agri["shadow_rapporteurs"] = df_agri["document_id"].apply(get_shadow_rapporteurs)


/var/folders/nd/td49stzx1sb1x3vskq7xg2540000gn/T/ipykernel_69459/2399249807.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_agri["shadow_rapporteurs"] = df_agri["document_id"].apply(get_shadow_rapporteurs)


In [34]:
df_agri

,document_id,title,rapporteurs,shadow_rapporteurs
0,2024/0027(COD),Granting equivalence with EU requirements to M...,VRECIONOVÁ Veronika (ECR),"BUDA Daniel (EPP), CÂRCIU Gheorghe (S&D), PENN..."
1,2023/0448(COD),Protection of animals during transport and rel...,"BUDA Daniel (EPP), METZ Tilly (Greens/EFA)","GIMÉNEZ LARRAZ Borja (EPP), NOICHL Maria (S&D)..."
2,2023/0447(COD),Welfare of dogs and cats and their traceability,VRECIONOVÁ Veronika (ECR),"DE MEO Salvatore (EPP), NARDELLA Dario (S&D), ..."
3,2023/0413(COD),Monitoring framework for resilient European fo...,"SARGIACOMO Eric (S&D), WIESNER Emma (Renew)","KÖHLER Stefan (EPP), BERNHUBER Alexander (EPP)..."
4,2023/0410(COD),Standing Forest and Forestry Expert Group,"TOVERI Pekka (EPP), WIESNER Emma (Renew)","BERNHUBER Alexander (EPP), TEMIDO Marta (S&D),..."
...,...,...,...,...
83,2023/2726(RPS),Commission Regulation amending Annex II to Reg...,RIVASI Michèle (Greens/EFA),None
84,2021/2608(RPS),Resolution on the draft Commission regulation ...,RIVASI Michèle (Greens/EFA),"SCHNEIDER Christine (EPP), HUITEMA Jan (Renew)"
85,2020/2795(RPS),Resolution on the draft Commission regulation ...,"NOVAK Ljudmila (EPP), ANDRIEU Eric (S&D), RIVA...",None
86,2020/2735(RPS),Resolution on the draft Commission regulation ...,"PIETIKÄINEN Sirpa (EPP), SCHALDEMOSE Christel ...",None
